# Day 3 — Hyperparameter tuning with GridSearchCV

**Project:** RetailMind AI — Customer Churn Prediction

**Goal:** Use cross-validation and GridSearchCV to tune selected classical models, then compare validation performance with the untouched test set.



Train several classical algorithms on the same training data after applying the shared preprocessing. Comparing Decision Tree, Random Forest and SVM helps determine whether a more flexible model captures churn patterns better than the initial Logistic Regression baseline. The metrics table makes the comparison consistent.

## 7. Day 3 — Hyperparameter tuning with GridSearchCV

Run GridSearchCV for the selected Decision Tree and Random Forest candidates. Each parameter combination is evaluated across multiple folds of the training data, and the search selects the configuration with the best cross-validated recall. The held-out test data is not used to choose hyperparameters.

In [7]:
# Tune a Decision Tree and Random Forest. GridSearchCV uses training data only.
day3_candidates = {
    "Decision Tree": (
        Pipeline([
            ("preprocessor", make_preprocessor(X_train)),
            ("classifier", DecisionTreeClassifier(
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        {
            "classifier__max_depth": [3, 5, 8, None],
            "classifier__min_samples_split": [2, 10, 20]
        }
    ),
    "Random Forest": (
        Pipeline([
            ("preprocessor", make_preprocessor(X_train)),
            ("classifier", RandomForestClassifier(
                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
            ))
        ]),
        {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 8, 12],
            "classifier__min_samples_split": [2, 5]
        }
    )
}

day3_searches = {}
day3_results = []

for name, (pipe, grid) in day3_candidates.items():
    print("Tuning:", name)
    search = GridSearchCV(
        pipe, grid, scoring="recall", cv=5, n_jobs=-1, refit=True
    )
    search.fit(X_train, y_train)
    day3_searches[name] = search

    best_model = search.best_estimator_
    metrics = evaluate_classifier(best_model, X_test, y_test, name + " Tuned")
    metrics["Best CV Recall"] = search.best_score_
    metrics["Best Parameters"] = str(search.best_params_)
    day3_results.append(metrics)

day3_comparison = pd.DataFrame(day3_results).sort_values(
    ["Best CV Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(day3_comparison)
day3_comparison.to_csv(REPORTS_DIR / "day3_tuning_results.csv", index=False)

day3_champion_name = day3_comparison.loc[0, "Model"].replace(" Tuned", "")
day3_champion = day3_searches[day3_champion_name].best_estimator_
joblib.dump(day3_champion, MODELS_DIR / "churn_v3.joblib")
print("Day 3 tuned champion:", day3_champion_name)


Tuning: Decision Tree
Tuning: Random Forest


,Model,Accuracy,Precision,Recall,ROC-AUC,Best CV Recall,Best Parameters
0,Random Forest Tuned,0.750177,0.519031,0.802139,0.840007,0.78796,"{'classifier__max_depth': 8, 'classifier__min_..."
1,Decision Tree Tuned,0.745919,0.514035,0.783422,0.814413,0.78194,"{'classifier__max_depth': 3, 'classifier__min_..."


Day 3 tuned champion: Random Forest


### Day 3 — Train versus validation scores

Compare the best cross-validation recall with the recall measured on the separate test set. A large gap can indicate that the model performs better on the folds used during tuning than on unseen data. The test score provides a more realistic check of how well the tuned model generalizes.

In [8]:
# Compare the best CV recall with held-out test recall as a practical diagnostic.
display(day3_comparison[["Model", "Best CV Recall", "Recall", "ROC-AUC"]])

print("Interpretation guide:")
print("- High training/CV performance with weaker validation/test performance can indicate overfitting.")
print("- Low scores on both training/CV and test can indicate underfitting.")
print("- A large gap is a warning sign; use cross-validation and learning curves for deeper diagnosis.")


,Model,Best CV Recall,Recall,ROC-AUC
0,Random Forest Tuned,0.78796,0.802139,0.840007
1,Decision Tree Tuned,0.78194,0.783422,0.814413


Interpretation guide:
- High training/CV performance with weaker validation/test performance can indicate overfitting.
- Low scores on both training/CV and test can indicate underfitting.
- A large gap is a warning sign; use cross-validation and learning curves for deeper diagnosis.


## Day 3 — Completion

This notebook contains the executable project workflow through Day 3. Run all cells in order and review the printed metrics, tables and plots before recording results in the project README.